# 03 · When is a detection unmatched?

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/blob/145a73e500cb06070f399a040206802cc279174e/content/03_matching_and_uncertainty.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/03_matching_and_uncertainty.ipynb` at `145a73e500cb06070f399a040206802cc279174e`; original SHA-256 `9602946ab6ebb9ddb35a6b832ed96784b1245b238dbc11752fd54d7884a6a577`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 03 · When is a detection unmatched?

[Run this notebook in your browser ↗](https://jltobias.github.io/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/lab/index.html?path=03_matching_and_uncertainty.ipynb)

**Evidence: synthetic positions and invented identifiers.** Explore how a matching rule creates an unmatched category. None of the positions identify real vessels.

Context: [GFW SAR product definitions](https://api-doc.globalfishingwatch.org/our-apis/documentation/docs/v3/4wings/). This transparent teaching algorithm is not GFW's matching algorithm.

Run the setup cell first. Browser Python downloads numerical packages on first use. Core exercises use bundled files; live access is optional. Export your notebook to preserve work.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "pandas", "matplotlib", "scipy", "plotly"])
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import oceanlab as lab
from IPython.display import display
print("Runtime:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)

## A crowded patch of sea
We invent ten AIS candidate positions and ten SAR detections. Positions are assumed to have been projected to acquisition time. `minute` represents the time offset of supporting evidence. Real analysis must propagate vessel motion and location uncertainty, including SAR displacement effects.

In [ ]:
sar, ais = lab.matching_demo()
MAX_KM, MAX_MINUTES = 3.0, 15.0
matched = lab.match_points(sar, ais, max_km=MAX_KM, max_minutes=MAX_MINUTES)
display(matched)
fig, ax = plt.subplots()
ax.scatter(ais.lon, ais.lat, marker="^", c="#087e83", label="Synthetic AIS candidates")
ax.scatter(sar.lon, sar.lat, marker="o", facecolors="none", edgecolors="#bd6e13", s=65, label="Synthetic SAR detections")
for row in matched.dropna(subset=["match"]).itertuples():
    a = ais[ais.id == row.match].iloc[0]
    ax.plot([row.lon,a.lon], [row.lat,a.lat], color="gray", lw=.8)
ax.set(xlabel="Longitude (°)", ylabel="Latitude (°)", title="SYNTHETIC MATCHING | one identity per detection")
ax.legend(); plt.show()

## Sensitivity is part of the result
The algorithm enforces one-to-one matches and allows an unmatched choice. It maximizes valid matches and then minimizes normalized distance-plus-time cost. It has no truth labels for the three added detections: a plausible correspondence is not a verified identity.

In [ ]:
rows=[]
for km in [.25,.5,1,2,3,5,10]:
    result=lab.match_points(sar,ais,max_km=km,max_minutes=MAX_MINUTES)
    rows.append({"distance_gate_km":km,"matched":result.match.notna().sum(),"unmatched":result.match.isna().sum()})
sensitivity=pd.DataFrame(rows)
display(sensitivity)
ax=sensitivity.plot(x="distance_gate_km",y=["matched","unmatched"],marker="o",color=["#087e83","#bd6e13"])
ax.set(ylabel="Synthetic detection count",title="SYNTHETIC SENSITIVITY | the rule changes the category")
plt.show()

## A defensible interpretation
Change the time gate while holding distance fixed. What happens when you require evidence within one minute? Could an apparently unmatched detection become matched with better reception? Explain why a larger gate can reduce unmatched counts while increasing false identity assignments.

**Do not conclude:** the unmatched count is a count of illegal vessels. **Report instead:** the number of candidates without a valid counterpart under the stated rule and evidence coverage.